# Episode 7: What “Learning” Means: Data, Models, Loss
<a href="https://colab.research.google.com/github/Kinvecta/how-llms-work/blob/main/notebooks/Episode%2007%20-%20Learning%20Loss.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

**Companion notebook** for the series *How Large Language Models Work: From Math to Transformers*.

This notebook reproduces the Math Lab numbers from Episode 7, one step per section. It uses only `numpy` (already installed in Colab), so choose **Runtime, Run all** and nothing else is needed.

Covers: mean squared error for Lines A and B, surprisal and the mean loss over N terms, the Gaussian constant check, and a grid check that minimizing the loss matches maximizing the likelihood. The data and probabilities are illustrative. The episode's one noise symbol is τ (the Gaussian standard deviation), because σ is reserved for the sigmoid.

Every value that appears in the episode is checked with an `assert`. If a cell fails, a number in the script and the math have drifted apart.

## Setup: the toy regression data

In [1]:
import numpy as np

u = np.array([1.0, 2.0, 3.0])
y = np.array([2.0, 4.0, 5.0])
N = len(u)

def predict(W, beta):
    return W * u + beta                      # y-hat = W u + beta, with W 1×1

def mse(W, beta):
    return float(np.mean((y - predict(W, beta)) ** 2))   # L = (1/N) sum (y - y-hat)^2

## Section 3: mean squared error for Lines A and B

In [2]:
# Line A: W = 2, beta = 0
predA = predict(2.0, 0.0)
print("Line A predictions:", predA, " errors:", y - predA, " MSE =", round(mse(2.0, 0.0), 4), "(1/3)")
assert np.allclose(predA, [2, 4, 6]) and np.allclose(y - predA, [0, 0, -1])
assert np.isclose(mse(2.0, 0.0), 1 / 3)

# Line B: W = 1.5, beta = 2/3 (the least-squares fit)
predB = predict(1.5, 2 / 3)
print("Line B predictions:", np.round(predB, 4), " = (13/6, 11/3, 31/6)   MSE =", round(mse(1.5, 2 / 3), 4), "(1/18)")
assert np.allclose(predB, [13 / 6, 11 / 3, 31 / 6])
assert np.isclose(mse(1.5, 2 / 3), 1 / 18) and round(mse(1.5, 2 / 3), 3) == 0.056

# Confirm Line B is the least-squares fit
X = np.column_stack([u, np.ones(N)])
(W_ls, b_ls), *_ = np.linalg.lstsq(X, y, rcond=None)
print("least squares:  W =", round(W_ls, 4), "  beta =", round(b_ls, 4))
assert np.isclose(W_ls, 1.5) and np.isclose(b_ls, 2 / 3)

Line A predictions: [2. 4. 6.]  errors: [ 0.  0. -1.]  MSE = 0.3333 (1/3)
Line B predictions: [2.1667 3.6667 5.1667]  = (13/6, 11/3, 31/6)   MSE = 0.0556 (1/18)
least squares:  W = 1.5   beta = 0.6667


## Section 4: surprisal and the mean loss
For one observed token, the cross-entropy is −ln p of that token (its **surprisal**). The loss is the mean over N terms. The N = 2 beat: “mat” at 0.30, a second observed token at 0.60.

In [3]:
surprisal = lambda p: -np.log(p)

terms = np.array([surprisal(0.30), surprisal(0.60)])
print("surprisals:", np.round(terms, 3), "nats")
L = terms.mean()
print("loss L = mean =", round(L, 3), "nats")
assert round(terms[0], 3) == 1.204 and round(terms[1], 3) == 0.511
assert round(L, 3) == 0.857

# Cross-entropy H(q, p) against a one-hot q keeps only one term
p_dist = np.array([0.30, 0.20, 0.15, 0.35])
q_onehot = np.array([1.0, 0.0, 0.0, 0.0])
H_qp = -np.sum(q_onehot[q_onehot > 0] * np.log(p_dist[q_onehot > 0]))
assert np.isclose(H_qp, surprisal(0.30))

surprisals: [1.204 0.511] nats
loss L = mean = 0.857 nats


## Math Lab, Step 4: the Gaussian case
With fixed-τ Gaussian noise, −ln p(y | u; θ) = ½ ln(2πτ²) + (y − ŷ)² / (2τ²). For τ = 1, subtract the constant and what is left is half the sum of squared errors.

In [4]:
tau = 1.0
def nll_terms(W, beta):
    resid = y - predict(W, beta)
    return 0.5 * np.log(2 * np.pi * tau ** 2) + resid ** 2 / (2 * tau ** 2)

const = 0.5 * np.log(2 * np.pi * tau ** 2)
for name, (W, b) in {"Line A": (2.0, 0.0), "Line B": (1.5, 2 / 3)}.items():
    nll = nll_terms(W, b).sum()
    left = nll - N * const                 # constant removed
    print(f"{name}: summed NLL = {nll:.3f}; minus the constant = {left:.3f}; half the sum of squares = {0.5 * np.sum((y - predict(W, b)) ** 2):.3f}")

assert round(nll_terms(1.5, 2 / 3).sum() - N * const, 3) == 0.083
assert round(nll_terms(2.0, 0.0).sum() - N * const, 3) == 0.5

Line A: summed NLL = 3.257; minus the constant = 0.500; half the sum of squares = 0.500
Line B: summed NLL = 2.840; minus the constant = 0.083; half the sum of squares = 0.083


## Math Lab, Step 3: the maximizer of the likelihood is the minimizer of the loss
A grid check on both toy models. This is a numerical illustration, not a proof.

In [5]:
# (a) Regression: grid over (W, beta). Compare argmin of MSE with argmax of the Gaussian likelihood.
Ws = np.linspace(0.0, 3.0, 301)
bs = np.linspace(-2.0, 3.0, 501)
mse_grid = np.empty((len(Ws), len(bs)))
loglik_grid = np.empty_like(mse_grid)
for i, W in enumerate(Ws):
    for j, b in enumerate(bs):
        mse_grid[i, j] = mse(W, b)
        loglik_grid[i, j] = -nll_terms(W, b).sum()      # ln of the likelihood

i1, j1 = np.unravel_index(np.argmin(mse_grid), mse_grid.shape)
i2, j2 = np.unravel_index(np.argmax(loglik_grid), loglik_grid.shape)
print(f"argmin MSE:         W = {Ws[i1]:.2f}, beta = {bs[j1]:.2f}")
print(f"argmax likelihood:  W = {Ws[i2]:.2f}, beta = {bs[j2]:.2f}")
assert (i1, j1) == (i2, j2)

argmin MSE:         W = 1.50, beta = 0.67
argmax likelihood:  W = 1.50, beta = 0.67


In [6]:
# (b) Categorical: observed tokens, a model p = (a, b, 1-a-b) over three words. Extra check, not in the episode script.
data = ["mat", "mat", "floor", "mat", "sofa"]            # illustrative
index = {"mat": 0, "floor": 1, "sofa": 2}
obs = np.array([index[t] for t in data])

best_loss, best_lik = (None, np.inf), (None, -np.inf)
grid = np.linspace(0.01, 0.98, 98)
for a in grid:
    for b in grid:
        c = 1 - a - b
        if c <= 0.0:
            continue
        p = np.array([a, b, c])
        mean_surprisal = np.mean(-np.log(p[obs]))        # L
        lik = np.prod(p[obs])                            # likelihood of the N terms
        if mean_surprisal < best_loss[1]:
            best_loss = ((a, b, c), mean_surprisal)
        if lik > best_lik[1]:
            best_lik = ((a, b, c), lik)

print("argmin of the mean surprisal:", np.round(best_loss[0], 2))
print("argmax of the likelihood    :", np.round(best_lik[0], 2))
assert np.allclose(best_loss[0], best_lik[0])
print("Both equal the observed frequencies (0.60, 0.20, 0.20) for this data.")

argmin of the mean surprisal: [0.6 0.2 0.2]
argmax of the likelihood    : [0.6 0.2 0.2]
Both equal the observed frequencies (0.60, 0.20, 0.20) for this data.


## Section 6: the myth check
Cross-entropy penalizes any probability below 1 on the observed token, even when that token is the model's top choice.

In [7]:
print(f"top choice at 0.45 -> surprisal = {surprisal(0.45):.3f} nats")
assert round(surprisal(0.45), 3) == 0.799

top choice at 0.45 -> surprisal = 0.799 nats


## Check question
A batch has N = 2 terms. The model gives the observed token probability 0.25 at the first term and 0.50 at the second. (a) Each term's surprisal? (b) L? (c) Does raising the first probability to 0.40 lower L?

In [8]:
t1, t2 = surprisal(0.25), surprisal(0.50)
L1 = (t1 + t2) / 2
t1b = surprisal(0.40)
L2 = (t1b + t2) / 2
print(f"(a) {t1:.3f} and {t2:.3f} nats")
print(f"(b) L = {L1:.3f} nats")
print(f"(c) yes: -ln 0.40 = {t1b:.3f}, so L = {L2:.3f} nats")
assert round(t1, 3) == 1.386 and round(t2, 3) == 0.693
assert round(L1, 3) == 1.040 and round(t1b, 3) == 0.916 and round(L2, 3) == 0.805
assert L2 < L1

(a) 1.386 and 0.693 nats
(b) L = 1.040 nats
(c) yes: -ln 0.40 = 0.916, so L = 0.805 nats


In [9]:
print("All Episode 7 checks passed.")

All Episode 7 checks passed.
